In [18]:
import os
import sys

# Clone the repository if it doesn't exist, or pull the latest changes
if not os.path.exists('/content/AutoDiff-Numpy'):
    !git clone https://github.com/Seydifa/AutoDiff-Numpy.git /content/AutoDiff-Numpy
else:
    %cd /content/AutoDiff-Numpy
    !git pull
    %cd /content

# Add the cloned directory to sys.path so we can import dnp
if '/content/AutoDiff-Numpy' not in sys.path:
    sys.path.insert(0, '/content/AutoDiff-Numpy')

import dnp
print("✅ Successfully cloned and imported dnp!")


Cloning into '/content/AutoDiff-Numpy'...
remote: Enumerating objects: 115, done.
remote: Counting objects: 100% (115/115), done.
remote: Compressing objects: 100% (85/85), done.
remote: Total 115 (delta 23), reused 114 (delta 22), pack-reused 0 (from 0)
Receiving objects: 100% (115/115), 1.78 MiB | 23.12 MiB/s, done.
Resolving deltas: 100% (23/23), done.
✅ Successfully cloned and imported dnp!


In [20]:
dir(dnp.core.layers)

['AvgPool2d',
 'BatchNorm1d',
 'BatchNorm2d',
 'Conv2d',
 'Dropout',
 'Flatten',
 'Linear',
 'MaxPool2d',
 'Module',
 'MultiHeadAttention',
 'ReLU',
 'ScaledDotProductAttention',
 'SelfAttention',
 'Sequential',
 'Sigmoid',
 'Softmax',
 'Tanh',
 'Tensor',
 '_ActivationModule',
 '__all__',
 '__builtins__',
 '__cached__',
 '__doc__',
 '__file__',
 '__loader__',
 '__name__',
 '__package__',
 '__spec__',
 'add',
 'avg_pool2d',
 'batch_norm',
 'dropout',
 'matmul',
 'max_pool2d',
 'np',
 'relu',
 'sigmoid',
 'softmax',
 'tanh']

In [19]:
import dnp.core.optimizers as optimizers
import dnp.core.layers as layers
import dnp.core.ops as ops

# Hyperparameters
batch_size = 32
block_size = 8
d_model = 32
num_heads = 4
num_layers = 2
learning_rate = 5e-3
max_iters = 100

def get_batch(split):
    data_source = train_data if split == 'train' else val_data
    ix = np.random.randint(len(data_source) - block_size, size=(batch_size,))
    x_np = np.stack([data_source[i:i+block_size] for i in ix])
    y_np = np.stack([data_source[i+1:i+block_size+1] for i in ix])
    return x_np, y_np

# --- 1. Define Model Architecture using our new built-in Layers ---
class FeedForward(layers.Module):
    def __init__(self, d_model):
        super().__init__()
        self.l1 = layers.Linear(d_model, 4 * d_model)
        self.relu = layers.ReLU()
        self.l2 = layers.Linear(4 * d_model, d_model)
        
    def forward(self, x):
        return self.l2(self.relu(self.l1(x)))

class TransformerBlock(layers.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.ln1 = layers.LayerNorm(d_model)
        self.attn = layers.MultiHeadAttention(d_model, num_heads)
        self.ln2 = layers.LayerNorm(d_model)
        self.ffwd = FeedForward(d_model)

    def forward(self, x, mask=None):
        x_ln1 = self.ln1(x)
        attn_out = self.attn(x_ln1, x_ln1, x_ln1, mask=mask)
        x = ops.add(x, attn_out)

        x_ln2 = self.ln2(x)
        ffwd_out = self.ffwd(x_ln2)
        x = ops.add(x, ffwd_out)
        return x

class TransformerLanguageModel(layers.Module):
    def __init__(self, vocab_size, d_model, block_size, num_heads, num_layers):
        super().__init__()
        self.tok_emb = layers.Embedding(vocab_size, d_model, name="TokEmb")
        self.pos_emb = layers.Embedding(block_size, d_model, name="PosEmb")
        
        self.blocks = []
        for i in range(num_layers):
            block = TransformerBlock(d_model, num_heads)
            self._modules[f"block_{i}"] = block
            self.blocks.append(block)
            
        self.ln_f = layers.LayerNorm(d_model)
        self.lm_head = layers.Linear(d_model, vocab_size, bias=False, name="LMHead")
        self.block_size = block_size

    def forward(self, idx):
        B, T = np.asarray(idx).shape
        
        tok_emb = self.tok_emb(idx)
        positions = np.arange(T)[None, :]
        pos_emb = self.pos_emb(positions)
        
        x = ops.add(tok_emb, pos_emb)
        causal_mask = np.triu(np.ones((T, T)) * -1e9, k=1)
        t_mask = dnp.core.Tensor(causal_mask)
        
        for block in self.blocks:
            x = block(x, mask=t_mask)
            
        x = self.ln_f(x)
        return self.lm_head(x)

# Setup
model = TransformerLanguageModel(vocab_size, d_model, block_size, num_heads, num_layers)
criterion = layers.CrossEntropyLoss()
optimizer = optimizers.AdamW(model.parameters(), lr=learning_rate)

# --- 2. Train the Model ---
print(f"--- Training for {max_iters} steps ---")
for step in range(max_iters):
    xb, yb = get_batch('train')
    logits = model(xb)
    loss = criterion(logits, yb)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    if step % 20 == 0 or step == max_iters - 1:
        print(f"Step {step:3d} | Loss: {loss.data:.4f}")

# --- 3. Generate Text ---
def generate_text(model, start_idx, max_new_tokens):
    idx = start_idx
    for _ in range(max_new_tokens):
        # crop context
        B, T = np.asarray(idx).shape
        idx_cond = idx if T <= block_size else idx[:, -block_size:]
        
        # forward pass
        logits = np.asarray(model(idx_cond))
        
        # select last step -> (B, V)
        logits_last = logits[:, -1, :] 
        
        # softmax manually for numpy array prediction
        exp_logits = np.exp(logits_last - np.max(logits_last, axis=-1, keepdims=True))
        probs = exp_logits / np.sum(exp_logits, axis=-1, keepdims=True)
        
        # sample
        idx_next = np.array([[np.random.choice(vocab_size, p=probs[0])]])
        idx = np.concatenate((idx, idx_next), axis=1)
        
    return idx

print("\n--- Generating some Shakespeare! ---")
context = np.array([[0]], dtype=np.int32) # Starting context is the 0-th token (\n)
generated_idx = generate_text(model, context, max_new_tokens=200)
print(decode(generated_idx[0].tolist()))


AttributeError: module 'dnp.core.layers' has no attribute 'Embedding'